In [ ]:
import random
from collections import deque
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from WarehouseEnv import DEFAULT_REWARD_CONFIG, WarehouseEnv

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

VARIANT_EPISODES = 500
ABLATION_EPISODES = 300
SMOKE_EPISODES = 40
MAX_STEPS_PER_EPISODE = 60


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_global_seed(SEED)
print(f"Using device: {DEVICE}")

## DQN Setup with Perception Mean Embeddings
This notebook trains three required variants on the warehouse task:
1. Vanilla DQN (no replay, no target network)
2. DQN + Experience Replay
3. DQN + Experience Replay + Target Network

It also includes a reward ablation pipeline using the best-performing setup.

In [ ]:
EXPECTED_CLASS_ORDER = ["Floor", "Wall", "Pallet", "Sign"]


def _normalize_class_names(names):
    return [str(name) for name in names]


def load_mean_embeddings(
    npz_path: str = "class_mean_embeddings_resnet18.npz",
    pt_path: str = "class_mean_embeddings_resnet18.pt",
):
    npz_file = Path(npz_path)
    pt_file = Path(pt_path)

    if npz_file.exists():
        payload = np.load(npz_file, allow_pickle=True)
        class_names = _normalize_class_names(payload["classes"].tolist())
        mean_embeddings = np.asarray(payload["mean_embeddings"], dtype=np.float32)
        source_path = str(npz_file)
    elif pt_file.exists():
        payload = torch.load(pt_file, map_location="cpu")
        class_names = _normalize_class_names(list(payload["classes"]))
        mean_embeddings = payload["mean_embeddings"].detach().cpu().numpy().astype(np.float32)
        source_path = str(pt_file)
    else:
        raise FileNotFoundError(
            f"Neither {npz_path} nor {pt_path} was found in the workspace root."
        )

    if mean_embeddings.ndim != 2:
        raise ValueError(f"Expected a 2D embedding matrix, got shape {mean_embeddings.shape}.")

    class_to_index = {name.lower(): idx for idx, name in enumerate(class_names)}
    missing = [name for name in EXPECTED_CLASS_ORDER if name.lower() not in class_to_index]
    if missing:
        raise ValueError(f"Missing expected classes in embedding payload: {missing}")

    ordered_rows = [class_to_index[name.lower()] for name in EXPECTED_CLASS_ORDER]
    ordered_embeddings = mean_embeddings[ordered_rows].astype(np.float32)

    if ordered_embeddings.shape[0] != 4:
        raise ValueError(
            f"Expected 4 class rows (Floor/Wall/Pallet/Sign), got {ordered_embeddings.shape[0]}."
        )

    if not np.all(np.isfinite(ordered_embeddings)):
        raise ValueError("Loaded embeddings contain non-finite values.")

    embeddings_dict = {idx: ordered_embeddings[idx] for idx in range(4)}
    return embeddings_dict, source_path


embeddings_dict, embedding_source = load_mean_embeddings()

# 0=Floor, 1=Wall, 2=Pallet, 3=Sign
grid_map = np.array(
    [
        [0, 0, 0, 1, 3],
        [0, 1, 0, 1, 0],
        [0, 1, 0, 0, 0],
        [0, 0, 0, 1, 0],
        [3, 1, 0, 2, 0],
    ],
    dtype=np.int64,
)

if not set(np.unique(grid_map)).issubset(set(embeddings_dict.keys())):
    raise ValueError("grid_map contains object IDs without corresponding embeddings.")

base_reward_config = dict(DEFAULT_REWARD_CONFIG)


def build_reward_config(reward_overrides=None):
    reward_cfg = dict(base_reward_config)
    if reward_overrides:
        reward_cfg.update(reward_overrides)
    return reward_cfg


def make_env(*, reward_overrides=None, random_start=True, start_pos=(0, 0)):
    return WarehouseEnv(
        grid_map=grid_map,
        embeddings_dict=embeddings_dict,
        reward_config=build_reward_config(reward_overrides),
        max_steps=MAX_STEPS_PER_EPISODE,
        start_pos=start_pos,
        random_start=random_start,
    )


def make_train_env(reward_overrides=None):
    return make_env(reward_overrides=reward_overrides, random_start=True)


def make_eval_env(reward_overrides=None):
    return make_env(reward_overrides=reward_overrides, random_start=False)


env_smoke = make_train_env()
obs, info = env_smoke.reset(seed=SEED)
print(f"Embedding source: {embedding_source}")
print(f"Observation shape: {obs.shape}, dtype: {obs.dtype}")
print(f"Reset info: {info}")
print(f"Reward config: {base_reward_config}")

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        indices = np.random.choice(len(self.buffer), size=batch_size, replace=False)
        states, actions, rewards, next_states, dones = zip(*(self.buffer[idx] for idx in indices))
        return (
            np.asarray(states, dtype=np.float32),
            np.asarray(actions, dtype=np.int64),
            np.asarray(rewards, dtype=np.float32),
            np.asarray(next_states, dtype=np.float32),
            np.asarray(dones, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)


class QNetwork(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)


@dataclass
class VariantConfig:
    name: str
    use_replay: bool
    use_target_network: bool


@dataclass
class TrainConfig:
    episodes: int = VARIANT_EPISODES
    max_steps: int = MAX_STEPS_PER_EPISODE
    gamma: float = 0.99
    lr: float = 1e-3
    batch_size: int = 64
    replay_capacity: int = 20_000
    target_update_interval: int = 200
    epsilon_start: float = 1.0
    epsilon_end: float = 0.05
    epsilon_decay_steps: int = 20_000
    hidden_dim: int = 128
    eval_episodes: int = 50
    log_interval: int = 100

In [ ]:
def epsilon_by_step(step: int, cfg: TrainConfig) -> float:
    if cfg.epsilon_decay_steps <= 0:
        return cfg.epsilon_end
    fraction = min(step / cfg.epsilon_decay_steps, 1.0)
    return cfg.epsilon_start + fraction * (cfg.epsilon_end - cfg.epsilon_start)


def select_action(state, q_network, epsilon: float, action_dim: int) -> int:
    if np.random.rand() < epsilon:
        return int(np.random.randint(action_dim))

    state_tensor = torch.as_tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    with torch.no_grad():
        q_values = q_network(state_tensor)
    return int(torch.argmax(q_values, dim=1).item())


def optimize_q_network(
    q_network,
    optimizer,
    loss_fn,
    states,
    actions,
    rewards,
    next_states,
    dones,
    gamma: float,
    target_network=None,
):
    states_t = torch.as_tensor(states, dtype=torch.float32, device=DEVICE)
    actions_t = torch.as_tensor(actions, dtype=torch.int64, device=DEVICE)
    rewards_t = torch.as_tensor(rewards, dtype=torch.float32, device=DEVICE)
    next_states_t = torch.as_tensor(next_states, dtype=torch.float32, device=DEVICE)
    dones_t = torch.as_tensor(dones, dtype=torch.float32, device=DEVICE)

    q_values = q_network(states_t).gather(1, actions_t.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        next_source = target_network if target_network is not None else q_network
        next_q_values = next_source(next_states_t).max(dim=1).values
        targets = rewards_t + gamma * (1.0 - dones_t) * next_q_values

    loss = loss_fn(q_values, targets)
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(q_network.parameters(), max_norm=5.0)
    optimizer.step()
    return float(loss.item())


def evaluate_policy(q_network, cfg: TrainConfig, reward_overrides=None, seed: int = SEED):
    env = make_eval_env(reward_overrides=reward_overrides)
    q_network.eval()

    rewards = []
    lengths = []
    successes = []

    with torch.no_grad():
        for episode in range(cfg.eval_episodes):
            state, _ = env.reset(seed=seed + 10_000 + episode)
            total_reward = 0.0

            for step in range(cfg.max_steps):
                state_tensor = torch.as_tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                action = int(torch.argmax(q_network(state_tensor), dim=1).item())
                next_state, reward, terminated, truncated, _ = env.step(action)

                total_reward += float(reward)
                state = next_state

                if terminated or truncated:
                    rewards.append(total_reward)
                    lengths.append(step + 1)
                    successes.append(1.0 if terminated else 0.0)
                    break
            else:
                rewards.append(total_reward)
                lengths.append(cfg.max_steps)
                successes.append(0.0)

    q_network.train()
    return {
        "avg_reward": float(np.mean(rewards)),
        "avg_length": float(np.mean(lengths)),
        "success_rate": float(np.mean(successes)),
    }


def build_variant_components(variant: VariantConfig, cfg: TrainConfig, state_dim: int, action_dim: int):
    q_network = QNetwork(state_dim, action_dim, hidden_dim=cfg.hidden_dim).to(DEVICE)
    target_network = None
    if variant.use_target_network:
        target_network = QNetwork(state_dim, action_dim, hidden_dim=cfg.hidden_dim).to(DEVICE)
        target_network.load_state_dict(q_network.state_dict())
        target_network.eval()

    optimizer = optim.Adam(q_network.parameters(), lr=cfg.lr)
    loss_fn = nn.SmoothL1Loss()
    replay_buffer = ReplayBuffer(cfg.replay_capacity) if variant.use_replay else None
    return q_network, target_network, optimizer, loss_fn, replay_buffer


def init_history():
    return {
        "episode_reward": [],
        "episode_length": [],
        "episode_success": [],
        "episode_loss": [],
    }


def update_online_or_replay(
    *,
    variant: VariantConfig,
    cfg: TrainConfig,
    state,
    action,
    reward,
    next_state,
    done,
    q_network,
    target_network,
    optimizer,
    loss_fn,
    replay_buffer,
 ):
    if variant.use_replay:
        replay_buffer.push(state, action, reward, next_state, done)
        if len(replay_buffer) < cfg.batch_size:
            return None
        batch = replay_buffer.sample(cfg.batch_size)
        return optimize_q_network(
            q_network=q_network,
            optimizer=optimizer,
            loss_fn=loss_fn,
            states=batch[0],
            actions=batch[1],
            rewards=batch[2],
            next_states=batch[3],
            dones=batch[4],
            gamma=cfg.gamma,
            target_network=target_network,
        )

    return optimize_q_network(
        q_network=q_network,
        optimizer=optimizer,
        loss_fn=loss_fn,
        states=np.asarray([state], dtype=np.float32),
        actions=np.asarray([action], dtype=np.int64),
        rewards=np.asarray([reward], dtype=np.float32),
        next_states=np.asarray([next_state], dtype=np.float32),
        dones=np.asarray([float(done)], dtype=np.float32),
        gamma=cfg.gamma,
        target_network=target_network,
    )


def run_training_episode(
    env,
    variant: VariantConfig,
    cfg: TrainConfig,
    q_network,
    target_network,
    optimizer,
    loss_fn,
    replay_buffer,
    global_step: int,
    episode_seed: int,
):
    state, _ = env.reset(seed=episode_seed)
    episode_reward = 0.0
    episode_losses = []

    for step in range(cfg.max_steps):
        epsilon = epsilon_by_step(global_step, cfg)
        action = select_action(state, q_network, epsilon, int(env.action_space.n))
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = bool(terminated or truncated)

        loss_value = update_online_or_replay(
            variant=variant,
            cfg=cfg,
            state=state,
            action=action,
            reward=reward,
            next_state=next_state,
            done=done,
            q_network=q_network,
            target_network=target_network,
            optimizer=optimizer,
            loss_fn=loss_fn,
            replay_buffer=replay_buffer,
        )
        if loss_value is not None:
            episode_losses.append(loss_value)

        state = next_state
        episode_reward += float(reward)
        global_step += 1

        if variant.use_target_network and global_step % cfg.target_update_interval == 0:
            target_network.load_state_dict(q_network.state_dict())

        if done:
            return {
                "episode_reward": episode_reward,
                "episode_length": step + 1,
                "episode_success": 1.0 if terminated else 0.0,
                "episode_loss": float(np.mean(episode_losses)) if episode_losses else 0.0,
                "global_step": global_step,
                "epsilon": epsilon,
            }

    return {
        "episode_reward": episode_reward,
        "episode_length": cfg.max_steps,
        "episode_success": 0.0,
        "episode_loss": float(np.mean(episode_losses)) if episode_losses else 0.0,
        "global_step": global_step,
        "epsilon": epsilon_by_step(global_step, cfg),
    }


def train_variant(
    variant: VariantConfig,
    cfg: TrainConfig,
    reward_overrides=None,
    seed: int = SEED,
):
    set_global_seed(seed)
    env = make_train_env(reward_overrides=reward_overrides)

    state_dim = int(env.observation_space.shape[0])
    action_dim = int(env.action_space.n)
    q_network, target_network, optimizer, loss_fn, replay_buffer = build_variant_components(
        variant, cfg, state_dim, action_dim
    )

    history = init_history()
    global_step = 0

    for episode in range(cfg.episodes):
        episode_metrics = run_training_episode(
            env=env,
            variant=variant,
            cfg=cfg,
            q_network=q_network,
            target_network=target_network,
            optimizer=optimizer,
            loss_fn=loss_fn,
            replay_buffer=replay_buffer,
            global_step=global_step,
            episode_seed=seed + episode,
        )
        global_step = episode_metrics["global_step"]

        history["episode_reward"].append(episode_metrics["episode_reward"])
        history["episode_length"].append(episode_metrics["episode_length"])
        history["episode_success"].append(episode_metrics["episode_success"])
        history["episode_loss"].append(episode_metrics["episode_loss"])

        if (episode + 1) % cfg.log_interval == 0 or episode == 0:
            recent_window = min(50, len(history["episode_reward"]))
            recent_reward = float(np.mean(history["episode_reward"][-recent_window:]))
            recent_success = float(np.mean(history["episode_success"][-recent_window:]))
            print(
                f"[{variant.name}] Episode {episode + 1}/{cfg.episodes} | "
                f"eps={episode_metrics['epsilon']:.3f} | mean_reward@{recent_window}={recent_reward:.3f} | "
                f"success@{recent_window}={recent_success:.3f}"
            )

    eval_metrics = evaluate_policy(
        q_network=q_network,
        cfg=cfg,
        reward_overrides=reward_overrides,
        seed=seed,
    )

    return {
        "variant": variant.name,
        "history": history,
        "eval": eval_metrics,
        "model": q_network,
    }


def run_variant_experiments(variant_configs, cfg: TrainConfig, seed: int = SEED, reward_overrides=None):
    results = {}
    for idx, variant in enumerate(variant_configs):
        print(f"\n=== Training {variant.name} ===")
        results[variant.name] = train_variant(
            variant=variant,
            cfg=cfg,
            reward_overrides=reward_overrides,
            seed=seed + idx * 100,
        )
    return results


def run_reward_ablations(reward_overrides_map, variant: VariantConfig, cfg: TrainConfig, seed: int = SEED):
    results = {}
    for idx, (name, override) in enumerate(reward_overrides_map.items()):
        print(f"\n=== Reward ablation: {name} ===")
        results[name] = train_variant(
            variant=variant,
            cfg=cfg,
            reward_overrides=override,
            seed=seed + 1_000 + idx * 100,
        )
    return results


def moving_average(values, window: int = 50):
    arr = np.asarray(values, dtype=np.float32)
    if arr.size == 0:
        return arr
    if arr.size < window:
        return arr
    kernel = np.ones(window, dtype=np.float32) / float(window)
    return np.convolve(arr, kernel, mode="valid")

## Variant Training (Required Comparison)
Runs the three assignment variants with identical hyperparameters and seeds.


In [ ]:
variant_configs = [
    VariantConfig(name="vanilla_dqn", use_replay=False, use_target_network=False),
    VariantConfig(name="dqn_replay", use_replay=True, use_target_network=False),
    VariantConfig(name="dqn_replay_target", use_replay=True, use_target_network=True),
]

train_config = TrainConfig(
    episodes=VARIANT_EPISODES,
    max_steps=MAX_STEPS_PER_EPISODE,
    gamma=0.99,
    lr=1e-3,
    batch_size=64,
    replay_capacity=20_000,
    target_update_interval=200,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay_steps=20_000,
    hidden_dim=128,
    eval_episodes=50,
    log_interval=100,
)

experiment_results = {}
RUN_FULL_VARIANT_TRAINING = True

if RUN_FULL_VARIANT_TRAINING:
    experiment_results = run_variant_experiments(
        variant_configs=variant_configs,
        cfg=train_config,
        seed=SEED,
        reward_overrides=None,
    )

    print("\nVariant evaluation summary:")
    for name, result in experiment_results.items():
        eval_metrics = result["eval"]
        print(
            f"{name:>18} | avg_reward={eval_metrics['avg_reward']:.3f} | "
            f"success_rate={eval_metrics['success_rate']:.3f} | "
            f"avg_length={eval_metrics['avg_length']:.2f}"
        )
else:
    print(
        "Set RUN_FULL_VARIANT_TRAINING = True when you want to execute the three comparison runs."
    )

In [ ]:
def plot_variant_results(results, smoothing_window: int = 50, title: str = "DQN Variant Comparison"):
    if not results:
        print("No results to plot. Run training first.")
        return

    names = list(results.keys())
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    for name in names:
        history = results[name]["history"]

        rewards = history["episode_reward"]
        reward_curve = moving_average(rewards, window=smoothing_window)
        reward_x = np.arange(len(reward_curve)) + (smoothing_window - 1 if len(rewards) >= smoothing_window else 0)
        axes[0].plot(reward_x, reward_curve, label=name)

        success = history["episode_success"]
        success_curve = moving_average(success, window=smoothing_window)
        success_x = np.arange(len(success_curve)) + (smoothing_window - 1 if len(success) >= smoothing_window else 0)
        axes[1].plot(success_x, success_curve, label=name)

        losses = history["episode_loss"]
        loss_curve = moving_average(losses, window=smoothing_window)
        loss_x = np.arange(len(loss_curve)) + (smoothing_window - 1 if len(losses) >= smoothing_window else 0)
        axes[2].plot(loss_x, loss_curve, label=name)

    axes[0].set_title("Episode Reward (smoothed)")
    axes[0].set_xlabel("Episode")
    axes[0].set_ylabel("Reward")

    axes[1].set_title("Success Rate (smoothed)")
    axes[1].set_xlabel("Episode")
    axes[1].set_ylabel("Success")

    axes[2].set_title("Training Loss (smoothed)")
    axes[2].set_xlabel("Episode")
    axes[2].set_ylabel("Loss")

    for ax in axes:
        ax.grid(True, alpha=0.3)

    axes[0].legend(loc="best")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


if experiment_results:
    plot_variant_results(experiment_results, smoothing_window=50)
else:
    print("Variant plots will appear after full variant training runs.")

In [ ]:
# Quick smoke run to validate end-to-end training logic before long runs.
smoke_variant = VariantConfig(
    name="smoke_replay_target",
    use_replay=True,
    use_target_network=True,
)

smoke_config = TrainConfig(
    episodes=SMOKE_EPISODES,
    max_steps=MAX_STEPS_PER_EPISODE,
    gamma=0.99,
    lr=1e-3,
    batch_size=32,
    replay_capacity=5_000,
    target_update_interval=100,
    epsilon_start=1.0,
    epsilon_end=0.10,
    epsilon_decay_steps=2_000,
    hidden_dim=128,
    eval_episodes=20,
    log_interval=20,
)

smoke_result = train_variant(
    variant=smoke_variant,
    cfg=smoke_config,
    reward_overrides=None,
    seed=SEED,
)

print("Smoke evaluation:", smoke_result["eval"])

## Reward Ablation Study
Ablation runs hold training hyperparameters fixed and remove one reward component at a time.
Recommended base variant for ablation: replay + target network.

In [ ]:
ablation_reward_overrides = {
    "full_reward": {},
    "no_step_penalty": {"step_penalty": 0.0},
    "no_wall_penalty": {"wall_penalty": 0.0},
    "no_progress_bonus": {"progress_bonus": 0.0},
}

ablation_variant = VariantConfig(
    name="ablation_replay_target",
    use_replay=True,
    use_target_network=True,
)

ablation_config = TrainConfig(
    episodes=ABLATION_EPISODES,
    max_steps=MAX_STEPS_PER_EPISODE,
    gamma=0.99,
    lr=1e-3,
    batch_size=64,
    replay_capacity=20_000,
    target_update_interval=200,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay_steps=20_000,
    hidden_dim=128,
    eval_episodes=50,
    log_interval=100,
)

ablation_results = {}
RUN_REWARD_ABLATION = True

if RUN_REWARD_ABLATION:
    ablation_results = run_reward_ablations(
        reward_overrides_map=ablation_reward_overrides,
        variant=ablation_variant,
        cfg=ablation_config,
        seed=SEED,
    )

    print("\nReward ablation evaluation summary:")
    for name, result in ablation_results.items():
        eval_metrics = result["eval"]
        print(
            f"{name:>18} | avg_reward={eval_metrics['avg_reward']:.3f} | "
            f"success_rate={eval_metrics['success_rate']:.3f} | "
            f"avg_length={eval_metrics['avg_length']:.2f}"
        )
else:
    print(
        "Set RUN_REWARD_ABLATION = True when you want to execute the reward ablation study."
    )

In [ ]:
def summarize_table(results):
    if not results:
        print("No results available yet.")
        return

    print(f"{'name':>20} | {'avg_reward':>10} | {'success_rate':>12} | {'avg_length':>10}")
    print("-" * 64)
    for name, result in results.items():
        metrics = result["eval"]
        print(
            f"{name:>20} | {metrics['avg_reward']:10.3f} | "
            f"{metrics['success_rate']:12.3f} | {metrics['avg_length']:10.2f}"
        )


print("\nVariant results table:")
summarize_table(experiment_results)

print("\nAblation results table:")
summarize_table(ablation_results)

In [ ]:
# Compact diagnostics: final metrics for interpretation
def _compact_eval(results_dict):
    out = {}
    for k, v in results_dict.items():
        eval_m = v.get("eval", {})
        hist = v.get("history", {})
        rewards = hist.get("episode_reward", [])
        success = hist.get("episode_success", [])
        lengths = hist.get("episode_length", [])
        losses = hist.get("episode_loss", [])
        out[k] = {
            "eval_avg_reward": round(float(eval_m.get("avg_reward", float("nan"))), 4),
            "eval_success_rate": round(float(eval_m.get("success_rate", float("nan"))), 4),
            "eval_avg_length": round(float(eval_m.get("avg_length", float("nan"))), 4),
            "train_last50_reward": round(float(np.mean(rewards[-50:])), 4) if rewards else None,
            "train_last50_success": round(float(np.mean(success[-50:])), 4) if success else None,
            "train_last50_length": round(float(np.mean(lengths[-50:])), 4) if lengths else None,
            "train_last50_loss": round(float(np.mean(losses[-50:])), 6) if losses else None,
            "episodes_logged": len(rewards),
        }
    return out

print("RUN_FULL_VARIANT_TRAINING:", RUN_FULL_VARIANT_TRAINING)
print("RUN_REWARD_ABLATION:", RUN_REWARD_ABLATION)
print("\nSmoke eval:")
print(smoke_result.get("eval", {}))

print("\nVariant compact metrics:")
print(_compact_eval(experiment_results))

print("\nAblation compact metrics:")
print(_compact_eval(ablation_results))

In [ ]:
from pathlib import Path

candidate_results = []
if isinstance(globals().get("experiment_results"), dict):
    for name, result in experiment_results.items():
        candidate_results.append(("experiment_results", name, result))
if isinstance(globals().get("smoke_result"), dict):
    smoke_name = smoke_result.get("variant", "smoke_replay_target")
    candidate_results.append(("smoke_result", smoke_name, smoke_result))

ranked_candidates = []
for source_group, name, result in candidate_results:
    if not isinstance(result, dict):
        continue

    model = result.get("model")
    eval_metrics = dict(result.get("eval", {}))
    history = dict(result.get("history", {}))
    rewards = history.get("episode_reward", [])
    successes = history.get("episode_success", [])
    lengths = history.get("episode_length", [])

    avg_reward = float(eval_metrics.get("avg_reward", float("-inf")))
    avg_length = float(eval_metrics.get("avg_length", float("inf")))
    success_rate = float(eval_metrics.get("success_rate", float("nan")))
    train_last50_reward = float(np.mean(rewards[-50:])) if rewards else float("-inf")
    train_last50_success = float(np.mean(successes[-50:])) if successes else float("nan")
    train_last50_length = float(np.mean(lengths[-50:])) if lengths else float("inf")

    if model is None:
        continue

    ranked_candidates.append(
        {
            "source_group": source_group,
            "variant": name,
            "avg_reward": avg_reward,
            "avg_length": avg_length,
            "success_rate": success_rate,
            "train_last50_reward": train_last50_reward,
            "train_last50_success": train_last50_success,
            "train_last50_length": train_last50_length,
            "model": model,
            "result": result,
        }
    )

if not ranked_candidates:
    raise RuntimeError(
        "No trained RL models were found in notebook globals. Run the training cells before exporting the checkpoint."
    )

eval_is_nondiscriminative = all(
    item["success_rate"] == 0.0 and item["avg_length"] >= float(MAX_STEPS_PER_EPISODE)
    for item in ranked_candidates
)

if eval_is_nondiscriminative:
    selection_metric = "train_last50_reward"
    ranked_candidates.sort(key=lambda item: item["train_last50_reward"], reverse=True)
    top_score = ranked_candidates[0]["train_last50_reward"]
    tied_variants = [
        item["variant"] for item in ranked_candidates if item["train_last50_reward"] == top_score
    ]
    selection_note = (
        "Selected by mean training reward over the last 50 episodes because evaluation metrics were non-discriminative "
        "(all success_rate=0.0 and avg_length=max_steps)."
    )
else:
    selection_metric = "eval_avg_reward"
    ranked_candidates.sort(key=lambda item: item["avg_reward"], reverse=True)
    top_score = ranked_candidates[0]["avg_reward"]
    tied_variants = [
        item["variant"] for item in ranked_candidates if item["avg_reward"] == top_score
    ]
    selection_note = "Selected by evaluation avg_reward."

best_entry = ranked_candidates[0]
best_model = best_entry["model"]
state_dim = int(best_model.net[0].in_features)
action_dim = int(best_model.net[-1].out_features)
hidden_dim = int(best_model.net[0].out_features)

if len(tied_variants) > 1:
    selection_note += f" Tied top variants for {selection_metric}: {tied_variants}. Selected the first encountered top-scoring policy."

policy_payload = {
    "selection_metric": selection_metric,
    "selection_score": top_score,
    "selection_note": selection_note,
    "tied_top_variants": tied_variants,
    "source_group": best_entry["source_group"],
    "variant": best_entry["variant"],
    "eval": {
        "avg_reward": best_entry["avg_reward"],
        "avg_length": best_entry["avg_length"],
        "success_rate": best_entry["success_rate"],
    },
    "train_last50": {
        "avg_reward": best_entry["train_last50_reward"],
        "success_rate": best_entry["train_last50_success"],
        "avg_length": best_entry["train_last50_length"],
    },
    "grid_map": np.asarray(grid_map, dtype=np.int64),
    "reward_config": dict(base_reward_config),
    "expected_class_order": list(EXPECTED_CLASS_ORDER),
    "embedding_source": str(embedding_source),
    "max_steps": int(MAX_STEPS_PER_EPISODE),
    "state_dim": state_dim,
    "action_dim": action_dim,
    "hidden_dim": hidden_dim,
    "seed": int(SEED),
    "state_dict": best_model.state_dict(),
}

export_path = Path("warehouse_dqn_best_avg_reward.pth")
torch.save(policy_payload, export_path)

print("RL policy ranking:")
for item in ranked_candidates:
    print(
        f"{item['variant']:>20} | source={item['source_group']:<18} | eval_avg_reward={item['avg_reward']:.4f} | "
        f"eval_avg_length={item['avg_length']:.2f} | success_rate={item['success_rate']:.4f} | "
        f"train_last50_reward={item['train_last50_reward']:.4f} | train_last50_success={item['train_last50_success']:.4f}"
    )
print(selection_note)
print(f"Saved {export_path.resolve()}")
print(
    {
        k: policy_payload[k]
        for k in ["variant", "selection_metric", "selection_score", "eval", "train_last50", "tied_top_variants"]
    }
)